# Create the Dataset for the Feature Reconstruction Model
Importing stuff...

In [1]:
from torchvision.models.video import swin3d_t, Swin3D_T_Weights ,r2plus1d_18, R2Plus1D_18_Weights
import os
import torch
import pandas as pd
import cv2
from torchvision import transforms
import torch.nn as nn
from PIL import Image

Get all the video paths:

In [2]:
main = 'some_videos_sample_sequential_clips'
actions = os.listdir(main)

video_paths = []

for action in actions:

    folder = os.path.join(main,action) # some_videos_sample_clips/action/
    folders = os.listdir(folder)

    for video in folders :
        file_path = os.path.join(folder, video) # some_videos_sample_clips/action/video

        clips = os.listdir(file_path)
        for clip in clips:
            clip_path = os.path.join(file_path, clip)

            video_paths.append(clip_path)

In [2]:
main = 'some_videos_sample_sequential_clips_2'
target_action = 'pull ups'

video_paths = []

folder = os.path.join(main, target_action)  # some_videos_sample_sequential_clips/riding mechanical bull/
videos = os.listdir(folder)

for video in videos:
    file_path = os.path.join(folder, video)  # some_videos_sample_sequential_clips/riding mechanical bull/video

    #video_paths.append(file_path)

    clips = os.listdir(file_path)
    for clip in clips:
        clip_path = os.path.join(file_path, clip)
        video_paths.append(clip_path)

We'll be using the Swin3D. It has better results than I3D and R2plus1D.

In [ ]:
model = r2plus1d_18(weights=R2Plus1D_18_Weights.KINETICS400_V1)
model.layer4 = nn.Identity()
model.avgpool = nn.Identity()
model.fc = nn.Identity()

In [4]:
model = swin3d_t(weights= Swin3D_T_Weights.KINETICS400_V1)
model.norm = nn.Identity()
model.head = nn.Identity()
#model.avgpool = nn.Identity()
model.features[6] = nn.Identity()

In [4]:
dummy_input = torch.randn(1, 3, 16, 224, 224)

model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
dummy_input = dummy_input.to(device)

with torch.no_grad():
    output = model(dummy_input)

print("Output shape:", output.shape)
print(f'Mean : {output[0].mean()} , std : {output[0].std()}')
print(len(output[0]))

Output shape: torch.Size([1, 768])
Mean : 9.3068927526474e-05 , std : 1.4505376815795898
768


In [5]:
def load_video_frames(video_path, num_frames=16):
    cap = cv2.VideoCapture(video_path)
    frames = []
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    for i in range(num_frames):
        frame_idx = int(i * (frame_count / num_frames))  # Sample evenly
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)  # Convert BGR to RGB
        frames.append(frame)

    cap.release()
    return frames

def preprocess_frames_2(frames):
    transform = transforms.Compose([
        transforms.Resize((256, 256)),  # Resize the frames
        transforms.CenterCrop(224),     # Crop the frames to 112x112
        transforms.ToTensor(),          # Convert frames to tensor (values scaled to [0, 1])
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),  # Normalize
        transforms.ConvertImageDtype(torch.float32)  # Convert to float32
    ])

    processed_frames = []
    for frame in frames:
        pil_frame = Image.fromarray(frame)
        
        # Apply the transformation (which includes scaling to [0, 1], normalizing, and converting to float32)
        processed_frame = transform(pil_frame)
        
        # Clip values to [0, 1] after normalization (if necessary)
        processed_frame = torch.clamp(processed_frame, 0, 1)

        processed_frames.append(processed_frame)

    # Stack frames → Shape: (T, C, H, W) → (16, 3, 112, 112)
    video_tensor = torch.stack(processed_frames)

    # Rearrange dimensions to (B, C, T, H, W) → (1, 3, 16, 112, 112)
    return video_tensor.permute(1, 0, 2, 3).unsqueeze(0)

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)

features = []
videos_path = []
labels = []
i = 0
for video in video_paths:

    print(f"{i + 1}/{len(video_paths)}", end='\r')
    i += 1
    frames = load_video_frames(video, num_frames=16)
    if len(frames) < 16:
        print("Warning: Not enough frames. Consider padding or skipping the video.")
    inputs = preprocess_frames_2(frames).to(device) 

    output = model(inputs)

    features.append(output.cpu().detach().numpy())
    videos_path.append(video)
    labels.append('normal')


data = {
    'feature_vector': features,
    'video_path': videos_path,
    'label': labels
}

df = pd.DataFrame(data)

pkl_file_path = 'swin3d_dataset_pull_ups.pkl'
df.to_pickle(pkl_file_path)